In [1]:
import numpy as np
import scipy
import scipy
import numpy as np
import glob
from cogent3 import get_app, open_data_store
from cogent3.maths.measure import jsd
import os
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots



In [2]:
def get_ingroup_names(triads_info):
    ingroup_names_dict = {}
    for identifier, info in triads_info.items():
        triads_names = info['triples_species_names']
        ingroup_names_dict[identifier] = [triads_names['ingroup1'], triads_names['ingroup2']]
    return ingroup_names_dict

def get_jsd_diff(triads_info):
    jsd_diff_dict = {}
    for identifier, info in triads_info.items():
        triads_info_value = info['triples_info_small_tree']
        nuc_freqs_dict = triads_info_value['nuc_freqs_dict']
        nuc_freq1 = nuc_freqs_dict['ingroup1']
        nuc_freq2 = nuc_freqs_dict['ingroup2']
        nuc_freq_internal_node = nuc_freqs_dict["internal_node"]
        jsd1 = jsd(nuc_freq1, nuc_freq_internal_node)
        jsd2 = jsd(nuc_freq2, nuc_freq_internal_node)
        jsd_diff = abs(jsd1 - jsd2)
        jsd_diff_dict[identifier] = jsd_diff
    return jsd_diff_dict

def get_jsd(triads_info):
    jsd_dict = {}
    for identifier, info in triads_info.items():
        triads_info_value = info['triples_info_small_tree']
        nuc_freqs_dict = triads_info_value['nuc_freqs_dict']
        nuc_freq1 = nuc_freqs_dict['ingroup1']
        nuc_freq2 = nuc_freqs_dict['ingroup2']
        nuc_freq_internal_node = nuc_freqs_dict["internal_node"]
        jsd1 = jsd(nuc_freq1, nuc_freq_internal_node)
        jsd2 = jsd(nuc_freq2, nuc_freq_internal_node)
        jsd_dict[identifier] = {'ingroup1': jsd1, 'ingroup2': jsd2}
    return jsd_dict

def get_ingroup_jsd(triads_info):
    ingroup_jsd_dict = {}
    for identifier, info in triads_info.items():
        triads_info_value = info['triples_info_small_tree']
        ingroup_jsd = triads_info_value['ingroup_jsd']
        ingroup_jsd_dict[identifier] = ingroup_jsd
    return ingroup_jsd_dict


def get_ingroup_ens_diff(triads_info):
    ens_ingroup_dict = {}
    for identifier, info in triads_info.items():
        triads_info_value = info['triples_info_small_tree']
        triads_names = info['triples_species_names']
        ens_dict = triads_info_value['ens']
        ens_ingroup = abs(np.log(ens_dict[triads_names['ingroup1']]/ ens_dict[triads_names['ingroup2']]))
        ens_ingroup_dict[identifier] = ens_ingroup
    return ens_ingroup_dict

def get_ingroup_ens_absdiff(triads_info):
    ens_ingroup_dict = {}
    for identifier, info in triads_info.items():
        triads_info_value = info['triples_info_small_tree']
        triads_names = info['triples_species_names']
        ens_dict = triads_info_value['ens']
        ens_ingroup = abs(ens_dict[triads_names['ingroup1']]- ens_dict[triads_names['ingroup2']])
        ens_ingroup_dict[identifier] = ens_ingroup
    return ens_ingroup_dict

def get_ens(triads_info):
    ens_value_dict = {}
    for identifier, info in triads_info.items():
        triads_info_value = info['triples_info_small_tree']
        triads_names = info['triples_species_names']
        ens_dict = triads_info_value['ens']
        ens1 = ens_dict[triads_names['ingroup1']]
        ens2 = ens_dict[triads_names['ingroup2']]
        ens_value_dict[identifier] = {'ingroup1': ens1, 'ingroup2': ens2}
    return ens_value_dict


def get_nabla_absdiff(triads_info):
    nabla_diff_dict = {}
    for identifier, info in triads_info.items():
        triads_info_value = info['triples_info_small_tree']
        triads_names = info['triples_species_names']
        nabla_dict = triads_info_value['nabla_values']
        nbala_diff = abs(nabla_dict[triads_names['ingroup1']] - nabla_dict[triads_names['ingroup2']])
        nabla_diff_dict[identifier] = nbala_diff
    return nabla_diff_dict

def remove_outliers_iqr(data1, data2):
    def compute_iqr_bounds(data):
        Q1 = np.percentile(data, 25)
        Q3 = np.percentile(data, 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 5 * IQR
        upper_bound = Q3 + 5 * IQR
        return lower_bound, upper_bound

    # Calculate IQR bounds for both lists
    lower_bound1, upper_bound1 = compute_iqr_bounds(data1)
    lower_bound2, upper_bound2 = compute_iqr_bounds(data2)

    # Filter out pairs where either value is an outlier
    filtered_data1 = []
    filtered_data2 = []

    for val1, val2 in zip(data1, data2):
        if (lower_bound1 <= val1 <= upper_bound1) and (lower_bound2 <= val2 <= upper_bound2):
            filtered_data1.append(val1)
            filtered_data2.append(val2)

    return filtered_data1, filtered_data2

In [3]:
import os

triples_model_fitting_dir = '/Users/gulugulu/clock/mammal_orthologs_hsap_1/triples_model_fitting'
gene_paths = glob.glob(os.path.join(triples_model_fitting_dir, '*/'))

In [4]:


# Function to load JSON data
def load_json_data(path):
    with open(path, 'r') as file:
        return json.load(file)
    
def remove_repested_names(gene_paths):
    species_names = {}
    for path in gene_paths:
        gene_name = os.path.basename(path.rstrip('/'))
        triads_data_path = os.path.join(path, 'triples_info_dict_new.json')
        triads_info = load_json_data(triads_data_path)
        ingroup_names = get_ingroup_names(triads_info)
        species_names[gene_name] = ingroup_names

    repeated_names_dict = {}
    for gene, species in species_names.items():
        repeated_names_dict[gene] = []  # Initialize a list to store pairs
        for identifier, ingroup in species.items():
            for identifier2, ingroup2 in species.items():
                if identifier < identifier2:  # Ensure each pair is only processed once
                    if ingroup == ingroup2:
                        repeated_names_dict[gene].append((identifier, identifier2))

    # Optional: Remove genes with no repeated pairs
    repeated_names_dict = {gene: pairs for gene, pairs in repeated_names_dict.items() if pairs}
    removed_identifier = {gene: [a[0] for a in repeated_names_dict[gene]] if gene in repeated_names_dict else [] for gene in species_names}
    return removed_identifier

def get_names(triads_info):
    names_dict = {}
    for identifier, info in triads_info.items():
        triads_names = info['triples_species_names']
        names_dict[identifier] = triads_names
    return names_dict

In [5]:
removed_identifier = remove_repested_names(gene_paths)

In [6]:
# Function to compute required values
def compute_values(path, removed_identifier):
    gene_name = os.path.basename(path.rstrip('/'))
    triads_data_path = os.path.join(path, 'triples_info_dict_new.json')
    triads_info_original = load_json_data(triads_data_path)
    triads_info = {k: v for k, v in triads_info_original.items() if k not in removed_identifier[gene_name]}
    ens_abs_diff_dict = get_ingroup_ens_absdiff(triads_info)
    jsd_diff_dict = get_jsd_diff(triads_info)
    ens_dict = get_ens(triads_info)
    jsd_dict = get_jsd(triads_info)
    ingroup_jsd_dict = get_ingroup_jsd(triads_info)
    nabla_absdiff_dict = get_nabla_absdiff(triads_info)
    species_names_dict = get_names(triads_info)
    ens_abs_diff_list = list(ens_abs_diff_dict.values())
    jsd_diff_list = list(jsd_diff_dict.values())
    ingroup_jsd_list = list(ingroup_jsd_dict.values())
    nabla_absdiff_list = list(nabla_absdiff_dict.values())
    ens_list = list(ens_dict.values())
    jsd_list = list(jsd_dict.values())
    species_names_list = list(species_names_dict.values())

    

    return  ens_abs_diff_list, jsd_diff_list, ingroup_jsd_list, nabla_absdiff_list, ens_list, jsd_list, species_names_list

In [7]:
# Initialize the dictionary to store the data
gene_data_dict = {}
# Populate the dictionary with data for each gene
for path in gene_paths:
    gene_name = os.path.basename(path.rstrip('/'))
    ens_abs_diff_list, jsd_diff_list, ingroup_jsd_list, nabla_absdiff_list, ens_list, jsd_list, species_names_list = compute_values(path, removed_identifier)
    gene_data_dict[gene_name] = {
        'ens_abs_diff': ens_abs_diff_list,
        'jsd_diff': jsd_diff_list, 
        'ingroup_jsd': ingroup_jsd_list,
        'nabla_absdiff': nabla_absdiff_list,
        'ens': ens_list,
        'jsd': jsd_list,
        'species_names': species_names_list
    }

# Spearman Correlation Test JAD Difference Vs. ENS difference

In [8]:
import pandas as pd
species_number_dict = {}
max_group_size_dict = {}
num_group_dict = {}

for gene, value in gene_data_dict.items():
    data_f = pd.DataFrame({
        'ens_abs_diff': np.sqrt(value['ens_abs_diff']),
        'jsd_diff': np.sqrt(value['jsd_diff']),
        'Species1': [x['ingroup1'] for x in value['species_names']],
        'Species2': [x['ingroup2'] for x in value['species_names']],
        'Species3': [x['outgroup'] for x in value['species_names']]
    })

    data_long = pd.melt(
        data_f,
        id_vars=['ens_abs_diff', 'jsd_diff'],
        value_vars=['Species1', 'Species2', 'Species3'],
        var_name='Species_Position',
        value_name='Species'
    )
    data_long['ens_abs_diff'] = data_long['ens_abs_diff']
    data_long['jsd_diff'] = data_long['jsd_diff']
    data_long['Species'] = data_long['Species'].astype(str)
    data_long['Species'] = data_long['Species'].astype('category')

    num_groups = len(set(data_long['Species']))
    species_number_dict[gene] = len(set(data_long['Species']))
    num_group_dict[gene] = num_groups
    max_group_size = max(data_long.groupby('Species').size())
    max_group_size_dict[gene] = max_group_size
    
correlation_list = {}
p_value_list = {}
for gene, lists in gene_data_dict.items():
    jsd_diff_list, ens_abs_diff_list = remove_outliers_iqr(lists['jsd_diff'], lists['ens_abs_diff'])

    #Add the correlation factor in the list
    cor, p_value = scipy.stats.spearmanr(jsd_diff_list, ens_abs_diff_list)
    correlation_list[gene] = cor
    p_value_list[gene] = p_value 

# Step 1: Correct p-values using the group size
corrected_p_value_jad = {gene: p_value_list[gene]*max_group_size_dict[gene] for gene in gene_data_dict.keys()}



significant_genes_corrected_5 = [gene for gene in gene_data_dict.keys() if corrected_p_value_jad[gene] < 0.05]
significant_genes_corrected_1 = [gene for gene in gene_data_dict.keys() if corrected_p_value_jad[gene] < 0.01]

significant_genes_correlation_dict_5 = {gene: correlation_list[gene] for gene in significant_genes_corrected_5}
significant_genes_correlation_dict_1 = {gene: correlation_list[gene] for gene in significant_genes_corrected_1}

# Step 2: Apply Benjamini-Hochberg procedure
# Create a DataFrame with genes and p-values
results_df = pd.DataFrame({
    'Gene': list(correlation_list.keys()),
    'Observed_Correlation': list(correlation_list.values()),
    'P_Value': list(corrected_p_value_jad.values())
})

# Remove genes with NaN p-values
results_df = results_df.dropna(subset=['P_Value'])

# Sort by p-value
results_df = results_df.sort_values('P_Value')

# Number of tests
m1 = len(results_df)

# Desired FDR level
alpha = 0.01

# Rank the p-values
results_df['Rank'] = np.arange(1, m1+1)

# Calculate the BH critical values
results_df['BH_Critical'] = results_df['Rank'] / m1 * alpha

# Determine significance
results_df['BH_Significant'] = results_df['P_Value'] <= results_df['BH_Critical']

# Find the largest p-value that is significant
significant_results1 = results_df[results_df['BH_Significant']]

if not significant_results1.empty:
    max_rank = significant_results1['Rank'].max()
    # All p-values up to max_rank are significant
    results_df['BH_Final_Significant'] = results_df['Rank'] <= max_rank
else:
    results_df['BH_Final_Significant'] = False



/var/folders/d8/pdrt51hx2jb17vf6k28_x6mh0000gn/T/ipykernel_85065/1264585017.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  max_group_size = max(data_long.groupby('Species').size())
/var/folders/d8/pdrt51hx2jb17vf6k28_x6mh0000gn/T/ipykernel_85065/1264585017.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  max_group_size = max(data_long.groupby('Species').size())
/var/folders/d8/pdrt51hx2jb17vf6k28_x6mh0000gn/T/ipykernel_85065/1264585017.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current beh

In [17]:

# Display significant results
significant_genes = results_df[results_df['BH_Significant']]

significant_correlated_genes = significant_genes[(significant_genes['Observed_Correlation'] > 0.2) & (significant_genes['P_Value'] < 0.05)]


In [18]:
len(significant_genes), len(significant_correlated_genes) #107/137, 97/107


(97, 97)

In [2]:
97/137

0.708029197080292

In [11]:
import plotly.express as px

# Data for histogram
values = list(correlation_list.values())
custom_colorscale = ['#F4A300', '#c73d47']

# Create the histogram with density normalization
fig2 = px.histogram(
    values,
    labels={'x': 'Correlation Coefficient', 'y': 'Density'},
    title=None,
    color_discrete_sequence=['#a7b8d8'],  # Set the color to a shade of orange
)   

# Update layout for presentation
fig2.update_layout(
    template='plotly_white',
    margin=dict(l=50, r=50, t=50, b=10),  # Adjust margins for a balanced look
    autosize=True,
    yaxis_title='<b>Count</b>',  # Explicit y-axis title
    xaxis_title=r'$\text{Spearman } \hat{\rho}$',  # Explicit x-axis title
    yaxis_title_font=dict(size=22),  # Adjust y-axis font size
    xaxis_title_font=dict(size=22),  # Adjust x-axis font size
    font=dict(size=20, color = 'black', family = 'Arial'),  # General font size for labels and titles
    width=800,  # Set figure width (optional for better control)
    height=400,  # Set figure height (optional for better control)
    showlegend=False  # Remove the legend
)

fig2.add_shape(
    type="line",
    x0=0.2, y0=0, x1=0.2, y1=15,
    line=dict(color="#c73d47", width=4, dash="dashdot"),
)

# Set transparency level and add a solid line around each bar
fig2.update_traces(
    opacity=1,  # Set the transparency (0 = fully transparent, 1 = fully opaque)
    marker_line_color='black',  # Color of the line around each bar
    marker_line_width=1.5,  # Width of the line around each bar
   xbins=dict(size=0.05)
)

fig2.update_xaxes(range=[-0.15, 0.9])



In [12]:

# Create the histogram with density normalization
fig2 = px.violin(
    list(correlation_list.values()),
    labels={'x': 'Correlation Coefficient', 'y': 'Density'},
    title=None,
    color_discrete_sequence=['#a7b8d8'],  # Set the color to a shade of orange
    orientation='h',
    points='all'
)   

# Update layout for presentation
fig2.update_layout(
    template='plotly_white',
    margin=dict(l=50, r=50, t=10, b=50),  # Adjust margins for a balanced look
    autosize=True,
    yaxis_title=None,  # Explicit y-axis title
    xaxis_title=r'$\text{Spearman } \hat{\rho}$',  # Explicit x-axis title
    xaxis_title_font=dict(size=22),  # Adjust x-axis font size
    font=dict(size=20, color = 'black', family = 'Arial'),  # General font size for labels and titles
    width=800,  # Set figure width (optional for better control)
    height=300,  # Set figure height (optional for better control)
    showlegend=False  # Remove the legend
)


fig2.add_shape(
    type="line",
    x0=0.2, y0=-0.5, x1=0.2, y1=0.5,
    line=dict(color="#c73d47", width=4, dash="dashdot"),
)

fig2.update_xaxes(
    showticklabels=True,
    showgrid=True,
    gridcolor='black',
    gridwidth=1,
    zeroline=True,
    zerolinecolor='black',
    zerolinewidth=1,
)

fig2.update_yaxes(showticklabels=False)




In [13]:
import pandas as pd
import plotly.express as px

# Custom color scale for categories
custom_colorscale = {
    'Pre-correction': '#fff4d3',  
    'Post-dependency correction': '#98c4ce', 
    'Post-dependency and multiple testing correction': '#e3edf7', 
}

# Create DataFrame
df = pd.DataFrame({
    "Category": ["Pre-correction", "Post-dependency correction", "Post-dependency and multiple testing correction"] * 2,
    "Significance Level": [0.01] * 3 + [0.05] * 3,
    "Significant Proportion": [125/137*100, 109/137*100, 107/137*100, 116/137*100, 102/137*100, 107/137*100],
    "Label": ["125/137", "109/137", "107/137", "116/137", "102/137", "107/136"]
})

# Code to create grouped bar chart using Plotly
fig = px.bar(
    df,
    x="Significance Level",
    y="Significant Proportion",
    color="Category",
    barmode="group",
    title=None,
    color_discrete_map=custom_colorscale,
    opacity=1,
    text="Label"  # Add text labels for each bar
)

fig.update_traces(
    textposition="outside",  # Place text outside the bars for better readability
    width=0.007,  # Adjust the width of the bars
    marker_line_color='black',  # Add a black line around the bars
    marker_line_width=2  # Set the width of the line around the bars
)

# Update layout for better formatting
fig.update_layout(
    template='plotly_white',
    margin=dict(l=50, r=50, t=5, b=50),
    autosize=True,
    yaxis_title='<b>Significant Correlation % </b>',
    xaxis_title='<b>Significance Level (α)</b>',
    font=dict(size=20, color='black', family='Arial'),
    width=600,
    height=395,
    showlegend=False,
    bargap=0.3,
    bargroupgap=0.1,
    legend=dict(
        title=None,
        font = dict(size=13),
        orientation="h",
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5
    ),

)

# Set text position to be inside the bars for readability
fig.update_traces(textposition="outside")

fig.show()

# # fig.write_image('/Users/gulugulu/repos/PuningAnalysis/results/figures/significant_proportion.pdf')


In [14]:
results_df

,Gene,Observed_Correlation,P_Value,Rank,BH_Critical,BH_Significant,BH_Final_Significant
87,ENSG00000118200,0.916294,4.410265e-78,1,0.000073,True,True
13,ENSG00000136643,0.878106,2.862116e-56,2,0.000146,True,True
55,ENSG00000118193,0.833954,1.474178e-52,3,0.000219,True,True
122,ENSG00000117114,0.840312,2.118443e-41,4,0.000292,True,True
25,ENSG00000162688,0.808301,1.187352e-38,5,0.000365,True,True
...,...,...,...,...,...,...,...
6,ENSG00000122218,0.085699,1.297065e+01,133,0.009708,False,False
36,ENSG00000058085,0.086919,2.294193e+01,134,0.009781,False,False
66,ENSG00000143624,0.051294,2.823414e+01,135,0.009854,False,False
29,ENSG00000133216,0.003253,5.673237e+01,136,0.009927,False,False


## Mixed linear model with random effect

In [ ]:


# def get_smf_mixedlm_result(gene, gene_data_dict):
#     # Step 1: Get the data for the gene
#     value = gene_data_dict[gene]
#     data_f = pd.DataFrame({
#         'ens_abs_diff': np.sqrt(value['ens_abs_diff']),
#         'jsd_diff': np.sqrt(value['jsd_diff']),
#         'Species1': [x['ingroup1'] for x in value['species_names']],
#         'Species2': [x['ingroup2'] for x in value['species_names']],
#         'Species3': [x['outgroup'] for x in value['species_names']]
#     })

#     data_long = pd.melt(
#         data_f,
#         id_vars=['ens_abs_diff', 'jsd_diff'],
#         value_vars=['Species1', 'Species2', 'Species3'],
#         var_name='Species_Position',
#         value_name='Species'
#     )

#     # Step 3: Adjust variables
#     data_long['ens_abs_diff'] = data_long['ens_abs_diff'] / 3
#     data_long['jsd_diff'] = data_long['jsd_diff'] / 3

#     # Step 4: Convert species identifiers to strings and factors
#     data_long['Species'] = data_long['Species'].astype(str)
#     data_long['Species'] = data_long['Species'].astype('category')

#     model = smf.mixedlm('ens_abs_diff ~ jsd_diff', data=data_long, groups=data_long['Species'])
#     result = model.fit()

#     var_random = result.cov_re.iloc[0, 0]

#     # Fixed effects variance
#     # Calculate variance of the linear predictor (fixed effects)
#     fixed_effects = result.fe_params
#     X = result.model.exog
#     var_fixed = np.var(np.dot(X, fixed_effects))

#     # Residual variance
#     var_residual = result.scale

#     # Marginal R-squared (fixed effects only)
#     R_m2 = var_fixed / (var_fixed + var_random + var_residual)

#     # Conditional R-squared (fixed + random effects)
#     R_c2 = (var_fixed + var_random) / (var_fixed + var_random + var_residual)

#     return result, R_m2, R_c2

In [ ]:
# # Example structure of your data
# import pandas as pd
# import statsmodels.formula.api as smf

# def get_smf_ols_result(gene, gene_data_dict):
#     # Step 1: Get the data for the gene
#     value = gene_data_dict[gene]
#     data_f = pd.DataFrame({
#         'ens_abs_diff': np.sqrt(value['ens_abs_diff']),
#         'jsd_diff': np.sqrt(value['jsd_diff']),
#         'Species1': [x['ingroup1'] for x in value['species_names']],
#         'Species2': [x['ingroup2'] for x in value['species_names']],
#         'Species3': [x['outgroup'] for x in value['species_names']]
#     })

#     data_long = pd.melt(
#         data_f,
#         id_vars=['ens_abs_diff', 'jsd_diff'],
#         value_vars=['Species1', 'Species2', 'Species3'],
#         var_name='Species_Position',
#         value_name='Species'
#     )

#     reduced_model = smf.ols('ens_abs_diff ~ jsd_diff', data=data_long)
#     result = reduced_model.fit()

#     return result

In [ ]:
# Example structure of your data
import pandas as pd
import statsmodels.formula.api as smf

multiple_linear_regression_data = {'gene': {}, 'p_value':{}, 'marginal_r^2': {}, 'conditional_r^2': {}}
for gene, value in gene_data_dict.items():
    data_f = pd.DataFrame({
        'ens_abs_diff': np.sqrt(value['ens_abs_diff']),
        'jsd_diff': np.sqrt(value['jsd_diff']),
        'Species1': [x['ingroup1'] for x in value['species_names']],
        'Species2': [x['ingroup2'] for x in value['species_names']],
        'Species3': [x['outgroup'] for x in value['species_names']]
    })

    data_long = pd.melt(
        data_f,
        id_vars=['ens_abs_diff', 'jsd_diff'],
        value_vars=['Species1', 'Species2', 'Species3'],
        var_name='Species_Position',
        value_name='Species'
    )

    # Step 3: Adjust variables
    data_long['ens_abs_diff'] = data_long['ens_abs_diff'] / 3
    data_long['jsd_diff'] = data_long['jsd_diff'] / 3

    # Step 4: Convert species identifiers to strings and factors
    data_long['Species'] = data_long['Species'].astype(str)
    data_long['Species'] = data_long['Species'].astype('category')

    model = smf.mixedlm('ens_abs_diff ~ jsd_diff', data=data_long, groups=data_long['Species'])
    result = model.fit()

        # Random effects variance
    var_random = result.cov_re.iloc[0, 0]

    # Fixed effects variance
    # Calculate variance of the linear predictor (fixed effects)
    fixed_effects = result.fe_params
    X = result.model.exog
    var_fixed = np.var(np.dot(X, fixed_effects))

    # Residual variance
    var_residual = result.scale

    # Marginal R-squared (fixed effects only)
    R_m2 = var_fixed / (var_fixed + var_random + var_residual)

    # Conditional R-squared (fixed + random effects)
    R_c2 = (var_fixed + var_random) / (var_fixed + var_random + var_residual)

    multiple_linear_regression_data['gene'][gene] = gene
    multiple_linear_regression_data['p_value'][gene] = result.pvalues['jsd_diff']
    multiple_linear_regression_data['marginal_r^2'][gene] = R_m2
    multiple_linear_regression_data['conditional_r^2'][gene] = R_c2

multiple_linear_regression_data['r^2_difference'] = {gene: multiple_linear_regression_data['conditional_r^2'][gene] - multiple_linear_regression_data['marginal_r^2'][gene] for gene in multiple_linear_regression_data['gene']}
r_squared_data = pd.DataFrame(multiple_linear_regression_data)

In [ ]:
r_squared_data

In [ ]:
import plotly.graph_objects as go

def plot_r2_by_gene(df):
    """
    df: pandas.DataFrame with columns 'gene', 'marginal_r^2', 'conditional_r^2'
    """
    genes = df['gene'].astype(str)  # categorical x-axis

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=genes, y=df['marginal_r^2'],
        mode='lines+markers',
        name='Marginal R²',
        marker=dict(color='#6fba4f', size=4),
        line=dict(width=1)
    ))
    fig.add_trace(go.Scatter(
        x=genes, y=df['conditional_r^2'],
        mode='lines+markers',
        name='Conditional R²',
        marker=dict(color='#e87cf4', size=4),
        line=dict(width=1)
    ))

    fig.update_layout(
        xaxis=dict(title='Gene', tickangle=-45),
        yaxis=dict(title='R²', range=[0, 1]),
        template='plotly_white',
        height=500,
        width = 1000,
        margin=dict(b=140)  # room for rotated gene labels
    )
    return fig

# Usage:
fig = plot_r2_by_gene(r_squared_data)
fig.show()


In [ ]:

import plotly.graph_objects as go
import pandas as pd
# Create horizontal Bar traces for Marginal R² and Conditional R²
trace_marginal = go.Bar(
    x=r_squared_data['marginal_r^2'],  # Swap the y-values to x for horizontal bars
    y=r_squared_data['gene'],  # Use gene names as y-axis values
    name='Marginal R²',
    marker_color='#6fba4f',
    hovertext=r_squared_data['gene'],  # Show gene names on hover
    hoverinfo='text+x',  # Display gene name and R² value on hover
    orientation='h'  # Set orientation to horizontal
)

trace_conditional = go.Bar(
    x=r_squared_data['conditional_r^2'],  # Swap the y-values to x for horizontal bars
    y=r_squared_data['gene'],  # Use gene names as y-axis values
    name='Conditional R²',
    marker_color='#e87cf4',
    hovertext=r_squared_data['gene'],  # Show gene names on hover
    hoverinfo='text+x',  # Display gene name and R² value on hover
    orientation='h'  # Set orientation to horizontal
)

# Combine the traces
data_traces = [trace_marginal, trace_conditional]

# Define the layout
layout = go.Layout(
    title=None,
    xaxis=dict(
        title='<b>R² Value<b>',  # X-axis now represents R² values
        tickfont=dict(size=14),
        range=[0, 1],  # Assuming R² ranges between 0 and 1
    ),
    yaxis=dict(
        title='<b>Gene</b>',  # Y-axis now represents genes
        showticklabels = False,  # Hide tick labels
    ),
    barmode='group',  # Side-by-side bars
    bargap=0.15,      # Gap between groups of bars
    bargroupgap=0.1,  # Gap between bars within a group
    legend=dict(
        x=0.85,
        y=1,
        bgcolor='rgba(255,255,255,0)',
        bordercolor='rgba(255,255,255,0)'
    ),
    template='plotly_white',
    margin=dict(l=60, r=30, t=80, b=60),
    height=1300  # Adjust height based on the number of genes to avoid squeezing
)

# Create the figure
fig = go.Figure(data=data_traces, layout=layout)

# Display the plot
fig.show()

# fig.write_image('/Users/gulugulu/repos/PuningAnalysis/results/figures/marginal_and_conditionalr_squared.pdf')


## Scatter plot for each gene

In [ ]:


# def plot_gene_data(gene_data_dict, xcol, ycol):
#     keys = list(gene_data_dict.keys())
#     rows = int(len(keys) ** 0.5) + 1  # Calculate the number of rows for subplots
#     cols = (len(keys) + rows - 1) // rows  # Calculate the number of columns

#     fig = make_subplots(rows=rows, cols=cols, subplot_titles=[f'{key}' for key in keys])
    
#     # Populate subplots
#     for index, key in enumerate(keys, start=1):
#         gene_data = gene_data_dict[key]
#         x_value, y_value = remove_outliers_iqr(gene_data[xcol], gene_data[ycol])

#         row = (index - 1) // cols + 1
#         col = (index - 1) % cols + 1
        
#         fig.add_trace(
#             go.Scatter(
#                 x=x_value,
#                 y=y_value,
#                 mode='markers',
#                 name=f'{key}'
#             ),
#             row=row,
#             col=col
#         )
        
#         # Adding a trend line
#         fig.add_trace(
#             go.Scatter(
#                 x=x_value,
#                 y=np.poly1d(np.polyfit(x_value, y_value, 1))(x_value),
#                 mode='lines',
#                 name=f'Trend {key}',
#                 line=dict(color='red')
#             ),
#             row=row,
#             col=col
#         )
        
#         # Update axis properties
#         fig.update_xaxes(title_text=xcol if row == rows else "", row=row, col=col)
#         fig.update_yaxes(title_text=ycol if col == 1 else "", row=row, col=col)
    
#     fig.update_layout(
#         height=300 * rows,  # Set a reasonable height based on the number of rows
#         width=300 * cols,   # Set a reasonable width based on the number of columns
#         showlegend=False
#     )
    
#     return fig

In [ ]:
# # Usage example
# fig = plot_gene_data(gene_data_dict, 'jsd_diff', 'ens_abs_diff')

# fig.update_layout(
# title_text="Scatter Plots of JAD Difference vs. ENS Difference",)

In [ ]:
path = '/Users/gulugulu/Desktop/honours/data_local_2/triples_model_fitting_550_threshold/ENSG00000065613'



In [ ]:
import os
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def get_triple_ens_diff_distirbution_plot(path):
    gene_name = path.split('/')[-1]
    triads_data_path = os.path.join(path, 'triples_info_dict.json')
    with open(triads_data_path, 'r') as f:
        triads_info = json.load(f)

    ens_ingroup_list = []
    ingroup_jsd_list = []
    for _, info in triads_info.items():
        triads_info_value = info['triples_info_small_tree']
        triads_names = info['triples_species_names']
        ingroup_jsd = triads_info_value['ingroup_jsd']
        ens_dict = triads_info_value['ens']
        ens_ingroup = abs(ens_dict[triads_names['ingroup1']] - ens_dict[triads_names['ingroup2']])
        ens_ingroup_list.append(ens_ingroup)
        ingroup_jsd_list.append(ingroup_jsd)

    # remove outliers (your function)
    ens_ingroup_list2, ingroup_jsd_list2 = remove_outliers_iqr(ens_ingroup_list, ingroup_jsd_list)

    # sort each list for clearer visualization and create indices
    ens_sorted = sorted(ens_ingroup_list2)
    jsd_sorted = sorted(ingroup_jsd_list2)
    idx_ens = list(range(1, len(ens_sorted) + 1))
    idx_jsd = list(range(1, len(jsd_sorted) + 1))

    # create 2-row vertical subplots
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.08,
        subplot_titles=(None, None)  # we will use overall title instead
    )

    # top: ENS difference
    fig.add_trace(
        go.Scatter(
            x=idx_ens, y=ens_sorted, mode='markers',
            marker=dict(size=4, color='#6fba4f'), name='ENS difference'
        ),
        row=1, col=1
    )

    # bottom: Ingroup JSD
    fig.add_trace(
        go.Scatter(
            x=idx_jsd, y=jsd_sorted, mode='markers',
            marker=dict(size=4, color='#f0a3f8'), name='Ingroup JSD'
        ),
        row=2, col=1
    )

    # axes labels
    fig.update_xaxes(title_text='<b>Index</b>', row=2, col=1)
    fig.update_yaxes(title_text='<b>ENS difference</b>', row=1, col=1)
    fig.update_yaxes(title_text='<b>Ingroup JSD</b>', row=2, col=1)

    # layout and sizing (tweak as needed)
    fig.update_layout(
        title=gene_name,
        showlegend=False,
        template='plotly_white',
        width=600,
        height=650,
        margin=dict(l=60, r=20, t=80, b=60)
    )

    return fig


In [ ]:
fig = get_triple_ens_diff_distirbution_plot(path)
fig.show()

In [ ]:
gene_name = path.split('/')[-1]

triads_data_path = os.path.join(path, 'triples_info_dict.json')
triads_info = load_json_data(triads_data_path)
ens_ingroup_list = []
ingroup_jsd_list = []
for identifier, info in triads_info.items():
    triads_info_value = info['triples_info_small_tree']
    triads_names = info['triples_species_names']
    ingroup_jsd = triads_info_value['ingroup_jsd']
    ens_dict = triads_info_value['ens']
    ens_ingroup = abs(ens_dict[triads_names['ingroup1']] - ens_dict[triads_names['ingroup2']])
    ens_ingroup_list.append(ens_ingroup)
    ingroup_jsd_list.append(ingroup_jsd)


ens_ingroup_list2, ingroup_jsd_list2 = remove_outliers_iqr(ens_ingroup_list, ingroup_jsd_list)


In [ ]:
indices_ens_diff2 = list(range(1, len(ens_ingroup_list2) + 1))
indices_ingroup_jsd2 = list(range(1, len(ingroup_jsd_list2) + 1))

list_pair = []
for i in range(len(indices_ens_diff2)):
    jsd_value, ens = ingroup_jsd_list2[i], ens_ingroup_list2[i]
    list_pair.append((jsd_value, ens))



In [ ]:
# Create the scatter plot
fig1 = go.Figure()

# # Add ENS Differences scatter plot
# fig1.add_trace(go.Scatter(
#     x=indices_ingroup_jsd2, y=sorted(ens_ingroup_list2), mode='markers',
#     marker=dict(size=4), name='ENS Differences'))

fig1.add_trace(go.Scatter(
    x=indices_ens_diff2, y=sorted(ens_ingroup_list2), mode='markers',
    marker=dict(size=4, color = '#6fba4f'), name='Ingroup JSD'))


# Update layout for clear visualization
fig1.update_layout(
    title=gene_name,
    xaxis_title='<b>Index</b>',
    yaxis_title='<b>ENS difference</b>',
    showlegend=False,
    template='plotly_white',
    margin=dict(l=20, r=20, t=50, b=20),
    width=600,
    height=300,    
)

fig1.show()
# fig1.write_image(f'/Users/gulugulu/repos/PuningAnalysis/results/figures/ENS_diff_{gene_name}_full.pdf')

In [ ]:
# Create the scatter plot
fig1 = go.Figure()

# # Add ENS Differences scatter plot
# fig1.add_trace(go.Scatter(
#     x=indices_ingroup_jsd2, y=sorted(ens_ingroup_list2), mode='markers',
#     marker=dict(size=4), name='ENS Differences'))

fig1.add_trace(go.Scatter(
    x=indices_ingroup_jsd2, y=sorted(ingroup_jsd_list2), mode='markers',
    marker=dict(size=4, color = "#f0a3f8"), name='Ingroup JSD'))


# Update layout for clear visualization
fig1.update_layout(
    title=gene_name,
    xaxis_title='<b>Index</b>',
    yaxis_title='<b>Ingroup JSD</b>',
    showlegend=False,
    template='plotly_white',
    margin=dict(l=20, r=20, t=50, b=20),
    width=600,
    height=300,    
)

# fig1.show()
# fig1.write_image(f'/Users/gulugulu/repos/PuningAnalysis/results/figures/ingroup_jsd_{gene_name}_full.pdf')